## Cookie fest

В данном соревновании, как вы уже увидели, вам предлагается решить задачу определения дубликатов товаров — необходимо определить, являются ли две товарные карточки одной и той же позицией.

В данных представлена публичная информация из карточки товара и первое изображение. Категория всех товаров на первом этапе — «Красота», что делает задачу особенно интересной: названия часто схожи, а описания могут быть неполными или не отражать специфику товара так хорошо как характеристики и изображения.

Этот бейзлайн служит отправной точкой и референсом, на основе которого можно строить собственные решения. Для простоты в решении не используются характеристики товара и изображения, однако, мы крайне рекомендуем обратить на эти данные особенное внимание. Подробное описание источников данных вы можете найти на странице соревнования. 

В конце ноутбука формируется файл сабмита, который можно загрузить в тестирующую систему для проверки. Используйте этот результат как референс и своё первое решение в соревновании.

## Imports and settings 

In [ ]:
from dataclasses import dataclass
from pathlib import Path
from pprint import pprint


import pandas as pd
import polars as pl
import torch
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split

In [3]:
@dataclass
class Config:
    
    #TODO Adjust
    DATA_DIR: Path = Path("data")
    # IMAGE_DIR: Path = Path("data/cleared/images/beauty")
    target: str = "target"
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # catboost settings 
    random_seed: int = 42
    cat_features: tuple[str] = ("parentname1", "parentname2", "subjectname1", "subjectname2")
    text_features: tuple[str] = ("title1", "title2", "description1", "description2")
    task_type = "GPU" if torch.cuda.is_available() else "CPU"
    verbose: int = 100
    eval_metric: str = "PRAUC"

cfg = Config()
print("Current device:", cfg.device)

Current device: cpu


## Data Loading

In [4]:
number_rows = 1000
df_train = pl.scan_parquet(cfg.DATA_DIR / "beauty_train.parquet",n_rows=number_rows)
# df_test  = pl.scan_parquet(cfg.DATA_DIR / "beauty_test.parquet",n_rows=number_rows)

print(f"Train num_rows: {df_train.select(pl.len()).collect().item():_}")
# print(f"Test num_rows: {df_test.select(pl.len()).collect().item():_}")

pprint(df_train.collect_schema())

Train num_rows: 1_000
Schema([('id', String),
        ('id1', String),
        ('parentname1', String),
        ('subjectname1', String),
        ('title1', String),
        ('description1', String),
        ('characteristics1',
         List(Struct({'value': Float64, 'charcName': String, 'charcValues': String}))),
        ('id2', String),
        ('parentname2', String),
        ('subjectname2', String),
        ('title2', String),
        ('description2', String),
        ('characteristics2',
         List(Struct({'value': Float64, 'charcName': String, 'charcValues': String}))),
        ('target', Boolean)])


In [5]:
# В базовом решении мы не используем информацию о характеристиках товаров
# df_train_modeling = df_test.collect().to_pandas()
# df_train_modeling.head(5)

df_transformed = (
    df_train
    # Преобразуем characteristics1
    .with_columns(
        pl.col("characteristics1")
        .list.eval(
            pl.format("{}: {} ({})", 
                      pl.element().struct.field("charcName"),
                      pl.element().struct.field("charcValues"),
                      pl.element().struct.field("value")
                     )
        )
        .alias("characteristics1")
    )
    # Преобразуем characteristics2
    .with_columns(
        pl.col("characteristics2")
        .list.eval(
            pl.format("{}: {} ({})", 
                      pl.element().struct.field("charcName"),
                      pl.element().struct.field("charcValues"),
                      pl.element().struct.field("value")
                     )
        )
        .alias("characteristics2")
    )
    # Теперь оба столбца — это list[str], старые struct автоматически заменены
)


In [60]:
pprint(df_transformed.collect_schema())

Schema([('id', String),
        ('id1', String),
        ('parentname1', String),
        ('subjectname1', String),
        ('title1', String),
        ('description1', String),
        ('characteristics1', List(String)),
        ('id2', String),
        ('parentname2', String),
        ('subjectname2', String),
        ('title2', String),
        ('description2', String),
        ('characteristics2', List(String)),
        ('target', Boolean)])


## Modeling

In [6]:
import embedding
emb = embedding.embedding()

c:\Users\User\kukihakaton\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Считаем эмбединги текстовых данных

In [7]:

# # Добавляем новую колонку 'embedding' в существующий датафрейм
# df_transformed = df_transformed.with_columns(
#     pl.col("title1")
#     .map_elements(emb.get_embeddings, return_dtype=pl.List(pl.Float64))
#     .alias("emb_title1")
# ).with_columns(
#     pl.col("description1")
#     .map_elements(emb.get_embeddings, return_dtype=pl.List(pl.Float64))
#     .alias("emb_description1")
# ).with_columns(
#     pl.col("characteristics1")
#     .map_elements(emb.get_embeddings, return_dtype=pl.List(pl.Float64))
#     .alias("emb_characteristics1")
# )

In [ ]:

# # Добавляем новую колонку 'embedding' в существующий датафрейм
# df_transformed = df_transformed.with_columns(
#     pl.col("title1")
#     .map_elements(emb.get_embeddings, return_dtype=pl.List(pl.Float64))
#     .alias("emb_title1")
# ).with_columns(
#     pl.col("description1")
#     .map_elements(emb.get_embeddings, return_dtype=pl.List(pl.Float64))
#     .alias("emb_description1")
# ).with_columns(
#     pl.col("characteristics1")
#     .map_elements(emb.get_embeddings, return_dtype=pl.List(pl.Float64))
#     .alias("emb_characteristics1")
# )

ДЕЛАЕМ ЭМБЕДДИНГ 

In [ ]:
embedding_exprs = []

columns = [    
    ("title1", "emb_title1"),
    ("description1", "emb_description1"),
    ("characteristics1", "emb_characteristics1"),
    ("title2", "emb_title2"),
    ("description2", "emb_description2"),
    ("characteristics2", "emb_characteristics2"),
]

for input_col, output_col in columns:
    expr = (
        pl.col(input_col)
        .map_batches(
            lambda series: pl.Series(
                values=[
                    emb.get_embeddings(text) if text is not None and isinstance(text, str) and len(text.strip()) > 0
                    else None  # или [0.0] * EMBEDDING_SIZE
                    for text in series
                ],
                return_dtype=pl.List(pl.Float64)
            )
        )
        .alias(output_col)
    )
    embedding_exprs.append(expr)

df_transformed = df_transformed.with_columns(embedding_exprs)

In [ ]:
# df_transformed = df_transformed.with_columns(
#     pl.col("title2")
#     .map_elements(emb.get_embeddings, return_dtype=pl.List(pl.Float64))
#     .alias("emb_title2")
# ).with_columns(
#     pl.col("description2")
#     .map_elements(emb.get_embeddings, return_dtype=pl.List(pl.Float64))
#     .alias("emb_description2")
# ).with_columns(
#     pl.col("characteristics2")
#     .map_elements(emb.get_embeddings, return_dtype=pl.List(pl.Float64))
#     .alias("emb_characteristics2")
# )

Считаем вектор сходства с помощью кос..четотам

In [ ]:
similarity_exprs = []
pairs = [
    ("title", "emb_title1", "emb_title2"),
    ("description", "emb_description1", "emb_description2"),
    ("characteristics", "emb_characteristics1", "emb_characteristics2"),
]

for name, col1, col2 in pairs:
    expr = (
        pl.struct([col1, col2])
        .map_batches(
            lambda s: pl.Series(
                emb.get_similarity(
                    emb1=s.struct.field(col1).to_list(),
                    emb2=s.struct.field(col2).to_list()
                ),
                return_dtype=pl.Float64
            )
        )
        .alias(f"{name}_sim")
    )
    similarity_exprs.append(expr)

df_transformed = df_transformed.with_columns(similarity_exprs)

In [ ]:
# df_transformed = df_transformed.with_columns(
#     pl.col(["emb_title1","emb_title2"])
#     .map_elements(emb.get_similarity, return_dtype=pl.List(pl.Float64))
#     .alias("title_sim")
# ).with_columns(
#     pl.col(["emb_description1","emb_description2"])
#     .map_elements(emb.get_similarity, return_dtype=pl.List(pl.Float64))
#     .alias("descr_sim")
# ).with_columns(
#     pl.col(["emb_characteristics1","emb_characteristics2"])
#     .map_elements(emb.get_similarity, return_dtype=pl.List(pl.Float64))
#     .alias("char_sim")
# )

In [10]:
pprint(df_transformed.collect_schema())

InvalidOperationError: UDF called without return type, but was not able to infer the output type.

This used to be allowed but lead to unpredictable results. To fix this problem, either provide a return datatype or execute the UDF in an eager context (e.g. in `map_columns`).

Resolved plan until failure:

	---> FAILED HERE RESOLVING 'with_columns' <---
 WITH_COLUMNS:
 [col("title1").python_udf().alias("emb_title1"), col("description1").python_udf().alias("emb_description1"), col("characteristics1").python_udf().alias("emb_characteristics1"), col("title2").python_udf().alias("emb_title2"), col("description2").python_udf().alias("emb_description2"), col("characteristics2").python_udf().alias("emb_characteristics2")] 
   WITH_COLUMNS:
   [col("characteristics2").list.eval(element().struct.field_by_name(charcName)().str.format([element().struct.field_by_name(charcValues)(), element().struct.field_by_name(value)()]))] 
     WITH_COLUMNS:
     [col("characteristics1").list.eval(element().str

In [ ]:
X = df_transformed.select(["emb_title1", "emb_description1", "emb_characteristics1"])

In [ ]:
X = df_train_modeling.drop(columns=["target", "id", "id1", "id2"])
y = df_train_modeling["target"]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y)

In [ ]:
train_pool = Pool(X_train, y_train, cat_features=cfg.cat_features, text_features=cfg.text_features)
val_pool = Pool(X_val, y_val, cat_features=cfg.cat_features, text_features=cfg.text_features)

model = CatBoostClassifier(
    verbose=cfg.verbose,
    eval_metric=cfg.eval_metric,
    task_type=cfg.task_type,
    random_state=cfg.random_seed
    )

model.fit(train_pool, eval_set=val_pool, early_stopping_rounds=100)

## Inference

In [ ]:
df_test_prediction = df_test.drop(["characteristics1", "characteristics2", "id", "id1", "id2"]).collect().to_pandas()

test_pool = Pool(df_test_prediction, cat_features=cfg.cat_features, text_features=cfg.text_features)
test_prediction = model.predict_proba(test_pool)[:, 1]

In [ ]:
sub_df = df_test.select("id").collect().to_pandas()
sub_df.loc[:, "y_pred"] = test_prediction
sub_df.to_csv(cfg.DATA_DIR / "sample_submission.csv", index=False)
sub_df.head()
# Sample Submission дает $/approx 0.916$ на публичном лидерборде